# nano-world-model — walkthrough

Train a **world model** you can play inside. Two small neural networks learn procgen *Chaser* from offline data:

1. a **VQ-VAE tokenizer** that turns each 64×64 frame into 64 discrete tokens ("the frame as 64 words"), and
2. a **GPT dynamics model** — structurally identical to [nanoGPT](https://github.com/karpathy/nanoGPT) — that predicts the next frame's tokens *conditioned on your action*.

There is no game engine at inference time. Every frame is sampled token-by-token from the GPT.

**Fast path** (~5 min): download pretrained checkpoints, roll out dreams, play with buttons.
**Slow path** (~2–4 h on a T4): train both models yourself — see the collapsed section near the end.

> While the repos are private, run `from huggingface_hub import login; login()` first.

In [ ]:
import os
if not os.path.exists("model.py"):          # running on Colab -> fetch the repo
    !git clone https://github.com/krohling/nanoGPT-WM.git
    %cd nanoGPT-WM
    %pip -q install -r requirements.txt

## 1. The data

Frames + actions from Meta's [gen_dgrl](https://github.com/facebookresearch/gen_dgrl) offline procgen release (PPO agents playing Chaser, expert + suboptimal mixed 50/50, CC-BY-NC 4.0), re-hosted in a dead-simple format: flat `uint8` binaries plus 60 complete held-out episodes. The fast path only needs the episodes (~24 MB).

In [ ]:
import numpy as np
import data

root = data.download(episodes_only=True)     # full dataset: data.download()
eps = data.list_episodes(root)
ep = data.load_episode(eps[0])
print(f"{len(eps)} held-out episodes; ep0: {ep['frames'].shape} actions {ep['actions'].shape}")

from eval import save_grid
save_grid(ep["frames"][::20][:8], "nb_frames.png", ncol=8)
from IPython.display import Image as _Img, display
display(_Img("nb_frames.png"))

## 2. Load the pretrained models

`tokenizer.pt` (VQ-VAE, 8×8 grid of 512-way codes) and `world_model.pt` (14M-param GPT, 16-frame context) — or point these paths at your own training outputs.

In [ ]:
import torch
from huggingface_hub import hf_hub_download
from eval import load_models

tok_path = hf_hub_download("kevin510/nano-world-model", "tokenizer.pt")
wm_path = hf_hub_download("kevin510/nano-world-model", "world_model.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
tok, wm = load_models(tok_path, wm_path, device)
print(f"tokenizer: {sum(p.numel() for p in tok.parameters())/1e6:.1f}M params, "
      f"{tok.tokens_per_frame} tokens/frame")
print(f"world model: {sum(p.numel() for p in wm.parameters())/1e6:.1f}M params, "
      f"context {wm.cfg.block_size} tokens")

### Tokenizer sanity: frames → 64 integers → frames

The only thing the GPT will ever see is the middle row of integers.

In [ ]:
from eval import to_uint8
x = torch.from_numpy(ep["frames"][:4].astype("float32") / 255).permute(0, 3, 1, 2).to(device)
idx = tok.encode(x)
recon = tok.decode(idx)
print("frame 0 as tokens:", idx[0].reshape(8, 8).cpu().numpy())
pair = np.concatenate([to_uint8(x), to_uint8(recon)], axis=2)
save_grid(pair, "nb_recon.png", ncol=4)
display(_Img("nb_recon.png"))

## 3. The world model is a language model

The training sequence interleaves frame tokens and action tokens —

```
z0¹ … z0⁶⁴  a0  z1¹ … z1⁶⁴  a1  z2¹ … z2⁶⁴ …
```

— one shared 527-word vocabulary (512 frame codes + 15 actions), one softmax, cross-entropy loss, autoregressive sampling. And the transformer itself is **verbatim nanoGPT**: `nanogpt.py` in this repo is Karpathy's `model.py`, vendored unchanged (diff it against upstream — the README shows the one-liner). Even the action-masking flows through nanoGPT's own `ignore_index=-1`. Each action token sits *between the frame where it was pressed and the frame it causes*, so generation is: append your action, sample 64 tokens, decode, repeat.

**FAQ — the four questions everyone asks** (long versions in the README):
1. *Why discrete tokens?* Regressing pixels averages over possible futures → ghost blurs. A softmax holds sharp alternatives and sampling **commits** to one.
2. *Why sample instead of argmax?* The world is stochastic; argmax per token can't coordinate a coherent outcome.
3. *Why autoregressive within a frame?* Parallel one-shot heads sample the product of marginals — attention shares *beliefs*, never sampled *outcomes*. An enemy at a junction goes half-left/half-right. Feeding each sampled token back in restores the joint (that's the chain rule).
4. *Why 64 tokens per frame, not 1?* One token per frame needs a codebook entry per distinguishable game state — combinatorially impossible. Spatial factorization expresses 512⁶⁴ scenes with 64×512 learned quantities.

## 4. Dream rollouts: real vs hallucinated

Prime with 4 real frames, then let the model dream forward following the *recorded* action sequence — top row real, bottom row dream (green separator = priming, red = hallucination).

In [ ]:
from eval import open_loop_rollout
from PIL import Image

P, H = 4, 40
dream = open_loop_rollout(tok, wm, ep["frames"], ep["actions"], P, H, 1.0, device)
real = ep["frames"][:P + H]
tiles = []
for t in range(P + H):
    sep = np.zeros((2, 64, 3), np.uint8)
    sep[:] = (0, 255, 0) if t < P else (255, 0, 0)
    tiles.append(np.concatenate([real[t], sep, dream[t]], axis=0))
imgs = [Image.fromarray(r).resize((192, 390), Image.NEAREST) for r in tiles]
imgs[0].save("nb_rollout.gif", save_all=True, append_images=imgs[1:], duration=120, loop=0)
display(_Img("nb_rollout.gif"))

## 5. Play inside the dream

Buttons drive the world model directly — every click samples the next frame from the GPT. (For real-time arrow-key play, run `python play.py …` locally; see README.) Watch the maze stay solid while *your* choices steer the agent. Raise the temperature and watch the world get unstable.

In [ ]:
import io
import ipywidgets as w
from eval import _crop_to_frames
from model import FRAME_VOCAB
from train_wm import interleave

K, SEQ_LEN = tok.tokens_per_frame, 16
state = {}

def to_png_bytes(frame):
    buf = io.BytesIO()
    Image.fromarray(frame).resize((320, 320), Image.NEAREST).save(buf, "png")
    return buf.getvalue()

def reset(_=None):
    e = data.load_episode(eps[int(np.random.randint(len(eps)))])
    x = torch.from_numpy(e["frames"][:4].astype("float32") / 255).permute(0, 3, 1, 2).to(device)
    toks = tok.encode(x).cpu().numpy().astype("int64")
    state["seq"] = interleave(toks, np.asarray(e["actions"][:4])).tolist()
    img.value = to_png_bytes(e["frames"][3])

def step(action):
    state["seq"] = _crop_to_frames(state["seq"], K, SEQ_LEN - 1)
    ctx = torch.tensor(state["seq"], device=device)[None]
    with torch.no_grad():
        nxt = wm.generate_frame(ctx, action, tokens_per_frame=K, temperature=temp.value)
    state["seq"] += [FRAME_VOCAB + action] + nxt[0].tolist()
    img.value = to_png_bytes(to_uint8(tok.decode(nxt))[0])

img = w.Image(format="png", width=320, height=320)
temp = w.FloatSlider(1.0, min=0.1, max=2.0, step=0.1, description="temp")
btns = {"←": 1, "↑": 5, "·": 4, "↓": 3, "→": 7}
row = []
for label, a in btns.items():
    b = w.Button(description=label, layout=w.Layout(width="48px"))
    b.on_click(lambda _, a=a: step(a))
    row.append(b)
rst = w.Button(description="reset"); rst.on_click(reset)
reset()
display(w.VBox([img, w.HBox(row + [rst]), temp]))

## 6. Train it yourself (optional — the slow path)

Two commands, straight out of nanoGPT's playbook. On a Colab T4: tokenizer ≲ 1 h, world model ~2–3 h (an A100 does both in ~40 min). Requires the full dataset download (~5.4 GB).

```bash
python -c "import data; data.download()"
python train_tokenizer.py --data data/chaser --out out/tok8 --steps 30000 --bs 128
python train_wm.py --data data/chaser --tokenizer out/tok8/tokenizer.pt --out out/wm --prepare
python train_wm.py --data data/chaser --tokenizer out/tok8/tokenizer.pt --out out/wm --steps 40000 --bs 16
```

Both scripts have `--overfit` sanity modes; `eval.py` makes the panels/GIFs/heatmaps shown in the README.

## 7. Things to try

- **Break the joint**: replace `generate_frame`'s loop with one-shot parallel prediction (take the top-1 of all 64 positions from a single forward pass) and find where the world falls apart.
- **Temperature sweep**: 0.1 → 2.0; when does the maze start flickering? Why does *some* temperature help vs. argmax?
- **Context ablation**: retrain with `--seq-len 2`; watch the dream lose object permanence (eaten orbs reappear).
- **16×16 tokens**: `--grid 16` in the tokenizer; better sprites, 4× longer sequences — feel the tradeoff from question 4 of the FAQ.
